# Proyecto Oráculo — Solución NPO (Llama 3.2 3B)

Análisis y Diseño de Algoritmos

Esta es una solución de ejemplo para la celda 4: implementa **Naive Prompt
Optimization (NPO)**, de Chang & Chen,
*Naive Prompt Optimization: Rethinking the Need for Complex Prompt Search*
([arXiv:2608.27266](https://arxiv.org/html/2608.27266)), con un teacher servido por
OpenRouter y el estudiante `llama3b` (`unsloth/Llama-3.2-3B-Instruct`).

```
r = oraculo.evaluar(config, instancias, semilla)
r_val = oraculo.validar(config, n)   # familias reservadas, muestra fija

r.precision        # 0.55
r.trazas           # [{id, violo, salida}, ...]
```


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo — `llama3b`

Antes: **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU**.
Sin GPU, `cargar_modelo` no arranca.

In [ ]:
from ayudas import cargar_modelo

modelo = cargar_modelo("llama3b")  # unsloth/Llama-3.2-3B-Instruct


## 3 · El oráculo

La corrida de NPO es larga (varios cientos de rollouts): montamos Drive para no
perder el caché si Colab se desconecta.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, RANURAS, TEMPERATURAS, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(
    modelo, busqueda, validacion,
    cache="/content/drive/MyDrive/cache_npo_llama3b.json",
)

CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")


## 4 · Naive Prompt Optimization (NPO)

Algoritmo 1 del paper — linaje único con ventana deslizante de trazas:

```
para i = 0..Y-1:
    muestrear minibatch B_i de tamaño N
    correr el estudiante con P(i) sobre B_i
    recolectar trazas de rollout y recompensas R_i
    ventana deslizante: {R_j} para j = max(0, i-W+1) .. i
    P(i+1) ← T(P(i), {R_j})       # T = teacher
devolver la secuencia de prompts y el mejor candidato
```

Lo que distingue a NPO de OPRO: el teacher recibe **trazas completas y recompensa por
rollout**, no solo prompts previos y un escalar.

En este proyecto el prompt no es texto libre: se arma escogiendo un índice por ranura
del `CATALOGO`. Por eso el teacher, en cada iteración, no reescribe texto sino que
propone la siguiente config del linaje — el resto del algoritmo (linaje único, minibatch
fresco, ventana deslizante, selección final en validación) se mantiene igual.

Hiperparámetros: los del paper para IFBench — `N=50`, `Y=10`, `N_VAL=300` (este oráculo
solo tiene 75 instancias de validación y las usa todas). El presupuesto de 3.500 rollouts
del paper sale de 50×10 (búsqueda) + 300×10 (validación): NPO valida **cada versión** del
linaje, no solo la última, y de ahí sale el mejor candidato.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  NPO — Naive Prompt Optimization  (arXiv:2608.27266, Algoritmo 1)
#  Linaje único + ventana deslizante de trazas. Teacher: OpenRouter.
#  Estudiante: el modelo cargado arriba, a través del oráculo.
# ═══════════════════════════════════════════════════════════════════════

import getpass
import json
import logging
import random
import re

import requests

logging.getLogger("absl").setLevel(logging.CRITICAL)  # ruido de langdetect en salidas sin palabras

# ─── Teacher ───────────────────────────────────────────────────────────
API_KEY = getpass.getpass("OpenRouter API key: ")
TEACHER = "openai/gpt-5.1"   # verifiquen el id en openrouter.ai/models


def teacher(mensaje):
    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "model": TEACHER,
            "messages": [{"role": "user", "content": mensaje}],
            "temperature": 1.0,
        },
        timeout=180,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


# ─── Hiperparámetros del paper (IFBench) ───────────────────────────────
N = 50            # tamaño del minibatch B_i
W = 3             # ventana deslizante (el paper la ajusta al contexto del teacher)
Y = 10            # iteraciones
N_VAL = 300       # validación del paper; este oráculo solo tiene 75 y las usa todas
SEMILLA = 1
MAX_TRAZAS = 8    # fallos que se le muestran al teacher por iteración
LARGO = 400       # chars de cada salida del estudiante


# ─── Armado del mensaje al teacher ─────────────────────────────────────
META = """Eres el modelo TEACHER de Naive Prompt Optimization (NPO).

Un modelo ESTUDIANTE resuelve tareas de seguimiento de instrucciones (estilo IFEval):
recibe una petición en inglés con restricciones verificables (número de palabras, formato,
frase final, mayúsculas, etc.). Un verificador automático da reward 1 si las cumple TODAS
y 0 si viola alguna.

El prompt del estudiante no es texto libre: se arma escogiendo un índice por ranura de este
catálogo fijo, y el texto elegido se antepone (o se pospone, en 'verificacion') a la petición.

CATÁLOGO
{catalogo}

Estas son las últimas {w} iteraciones de la optimización, cada una con su config, su
recompensa media y las trazas de los rollouts que fallaron:

{ventana}

Configs ya probadas (evita repetirlas sin una buena razón):
{probadas}

Tu trabajo: proponer la SIGUIENTE config del linaje, razonando sobre POR QUÉ fallaron esos
rollouts concretos (mira la restricción violada y el texto que produjo el estudiante).
Responde SOLO con un objeto JSON, sin markdown ni texto alrededor:
{{"razon": "<una frase>", "rol": <int>, "estrategia": <int>, "formato": <int>,
  "verificacion": <int>, "temperatura": <float>}}"""


def _catalogo_txt():
    lineas = []
    for ranura in RANURAS:
        lineas.append(f"{ranura}:")
        for i, texto in enumerate(CATALOGO[ranura]):
            lineas.append(f"  {i} = {texto!r}" if texto else f"  {i} = (vacío)")
    lineas.append("temperatura: " + ", ".join(str(t) for t in TEMPERATURAS))
    return "\n".join(lineas)


def _feedback(paso):
    """Una iteración de la ventana: config + recompensa + trazas de rollout."""
    p = [
        f"### Iteración {paso['i']}",
        f"config: {json.dumps(paso['config'])}",
        f"recompensa media: {paso['precision']:.3f}"
        f"  ({paso['n'] - len(paso['trazas'])}/{paso['n']} rollouts con reward 1)",
    ]
    if paso["trazas"]:
        p.append("rollouts con reward 0:")
        for t in paso["trazas"][:MAX_TRAZAS]:
            p.append(f"- restricción violada: {t['violo']}")
            p.append(f"  salida del estudiante: {t['salida'][:LARGO]!r}")
    else:
        p.append("ningún fallo en este minibatch.")
    return "\n".join(p)


def _leer_config(texto, anterior):
    """JSON del teacher → config válida. Ante cualquier duda, deja la anterior."""
    m = re.search(r"\{.*\}", texto, re.S)
    if not m:
        return anterior, "(el teacher no devolvió JSON)"
    try:
        d = json.loads(m.group())
    except json.JSONDecodeError:
        return anterior, "(JSON inválido)"

    nueva = {}
    for ranura in RANURAS:
        i = d.get(ranura, anterior[ranura])
        nueva[ranura] = (
            i if isinstance(i, int) and 0 <= i < len(CATALOGO[ranura]) else anterior[ranura]
        )
    t = d.get("temperatura", anterior["temperatura"])
    nueva["temperatura"] = t if t in TEMPERATURAS else anterior["temperatura"]
    return nueva, str(d.get("razon", ""))


# ─── Algoritmo 1 ───────────────────────────────────────────────────────
config = {"rol": 0, "estrategia": 0, "formato": 0, "verificacion": 0, "temperatura": 0.0}

historia, historial, probadas = [], [], []

for i in range(Y):
    lote = random.Random(SEMILLA + i).sample(busqueda, N)      # minibatch B_i
    r = oraculo.evaluar(config, lote, semilla=SEMILLA)         # rollouts + recompensas R_i

    historia.append(
        {"i": i, "config": config, "precision": r.precision, "trazas": r.trazas, "n": r.n}
    )
    historial.append(r.precision)
    probadas.append(dict(config))
    print(f"[{i}] recompensa {r.precision:5.1%}   {config}")

    if i == Y - 1:
        break

    ventana = "\n\n".join(_feedback(p) for p in historia[-W:])  # {R_j}, j = i-W+1 .. i
    mensaje = META.format(
        catalogo=_catalogo_txt(),
        w=min(W, len(historia)),
        ventana=ventana,
        probadas=json.dumps(probadas, ensure_ascii=False),
    )
    try:
        config, razon = _leer_config(teacher(mensaje), config)  # P(i+1) ← T(P(i), {R_j})
        print(f"     teacher: {razon}")
    except Exception as e:
        print(f"     teacher falló ({e}) — se repite la config")


# ─── Mejor candidato: cada versión del linaje, medida en validación ────
#  Presupuesto del paper en IFBench: 50*10 (búsqueda) + 300*10 (validación) = 3.500.
mejor = None
for paso in historia:
    print(f"\n[{paso['i']}] {paso['config']}")
    rv = oraculo.validar(paso["config"], n=N_VAL)
    paso["validacion"] = rv.precision
    if mejor is None or rv.precision > mejor[0]:
        mejor = (rv.precision, paso["config"])

print(f"\nmejor config: {mejor[1]}   validación {mejor[0]:.1%}")
print(f"presupuesto: {N * Y} rollouts de búsqueda + {Y} validaciones")


### Leer los fallos de la mejor config

Cada resultado trae sus trazas: mírenlas todas las veces que quieran.

In [ ]:
r = oraculo.evaluar(mejor[1], busqueda[:50], semilla=1)

for t in r.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)


### Curva del linaje

Recompensa de minibatch por iteración vs. la validación de cada versión — así se ve
si el linaje mejora de verdad o si el minibatch solo fue más fácil esa vez.

In [ ]:
from ayudas import curva

curva(historial)  # mejor recompensa de minibatch, acumulada

for paso in historia:
    print(f"[{paso['i']}]  minibatch {paso['precision']:5.1%}   validación {paso['validacion']:5.1%}")


## 5 · La entrega

In [ ]:
from ayudas import entrega
from google.colab import files

entrega(grupo="G07", config=mejor[1], semana=3)
files.download("entrega.json")
